# Experiment 7: recruitment, containment, and family response breadth

This notebook quantifies recruitment, signed response-window activity, containment
in control and all rocking variants, family response breadth, and exact stimulus
combinations. Summary panels are displayed **left then right**, with matching
scales. Original section numbers are retained for reference.

This notebook runs independently of notebook 01, reusing its cohort, portable
loading, timing, and activity criteria. Analyses preserve kept-neuron identities
and weight fish equally. Shared preparation supplies the response populations
required by the breadth helper; similarity, continuous-profile, and sparseness
figures are omitted from this notebook and its exports.

R300, R600, RF600, F300, F600, and DF300 denote **stimulus variants**. Interpret
the plots descriptively, without assuming an anatomical hierarchy, a
frequency-only manipulation, or biological equivalence. Independent-trial
preference validation remains deferred.


## 1. Setup

Find the repository and import the shared loading, analysis, figure, and reporting owners.


In [ ]:
from pathlib import Path
import sys
import importlib
from IPython.display import display
import matplotlib.pyplot as plt

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.plotting_common as plot_common
import src.plotting_diagnostics as plotting_diagnostics
import src.stimulus_specificity as stimulus_specificity
import src.neuron_ordering as neuron_ordering
import src.plotting_specificity as plotting_specificity
import src.several_fish_loading as several_loading
import src.several_fish_diagnostics as several_diagnostics
import src.several_fish_figures as several_figures
import src.several_fish_reporting as several_reporting
# Reload changed dependencies before modules that import their functions.
for module in (plot_common, plotting_diagnostics, stimulus_specificity, neuron_ordering, plotting_specificity,
               several_loading, several_diagnostics, several_reporting, several_figures):
    importlib.reload(module)


## Publication layout and export

The notebook uses **6 pt Arial** at final print size. Sections **5, 7, 13 and 14**
each produce four independent **45 × 27.5 mm** figures. Individual fish are gray
dots; the equally weighted fish mean is a black line. Legends show only “Fish”
and “Mean.” Width and height controls remain beside every plotting call.

Each panel has its own title and axis labels. Left/right scales and the configured
stimulus order are preserved. Set `EXPORT_PUBLICATION_FIGURES=True` and run the
export cell after review to write editable PDF/SVG and 600 dpi PNG. Exported
canvas dimensions match the configured millimetres without tight cropping.


In [ ]:
# Physical sizes are millimetres; font sizes are points at the final print size.
MM = 1 / 25.4
PUBLICATION_STYLE = {
    'font.family': 'Arial', 'font.size': 6,
    'axes.labelsize': 6, 'axes.titlesize': 6, 'figure.titlesize': 6,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6,
    'legend.title_fontsize': 6, 'axes.linewidth': 0.6,
    'lines.linewidth': 1.0, 'lines.markersize': 3,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'figure.dpi': 150, 'savefig.dpi': 600, 'savefig.bbox': None,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'mathtext.fontset': 'dejavusans', 'figure.facecolor': 'white',
}
plt.rcParams.update(PUBLICATION_STYLE)
EXPORT_PUBLICATION_FIGURES = False  # Enable after reviewing the figures.
PUBLICATION_FORMATS = ('pdf', 'svg', 'png')
PUBLICATION_DPI = 600
PUBLICATION_DIR = repo_root / 'notebooks/calcium/exp_07_trans_stat/publication_figures/02'


## 2. Cohort, mirrored stimulus groups, and activity criteria

Keep experiment-specific choices here. The activity criteria match notebook 01's overlap/count figures. `data_base=None` discovers OneDrive automatically; a manual override is the parent of the university OneDrive folder.


In [ ]:
SIDE_GROUPS = {
    'left': {
        'control': 'LeBcontrol',
        'rocking': ['R300Le', 'R600Le', 'RF600Le'],
        'flicker': ['F300Le', 'F600Le', 'DF300Le'],
    },
    'right': {
        'control': 'RiBcontrol',
        'rocking': ['R300Ri', 'R600Ri', 'RF600Ri'],
        'flicker': ['F300Ri', 'F600Ri', 'DF300Ri'],
    },
}
CONDITION_LABELS = ['Control', 'R300', 'R600', 'RF600', 'F300', 'F600', 'DF300']
USER_SETTINGS = {
    'experiment_name': 'Exp_7_trans_stat',
    'analysis_label': 'Exp 7 recruitment and containment',
    'data_base': None,
    'fish_ids': ['L742_f03', 'L742_f05', 'L742_f06', 'L742_f09', 'L742_f10'],
    'selected_blocks': ['B1', 'B2', 'B3', 'B4'],
    'verbose_loading': False,
    'timing': {'fps_2p': 2.0, 't_pre_s': 5.0, 't_post_s': 25.0,
               'motion_onset_s': 8.0, 'tau_s': 6.0},
    'stim_order': [7, 8, 10, 12, 3, 5, 1, 14, 9, 11, 13, 4, 6, 2],
}
ACTIVE_SETTINGS = {
    'active_fraction_threshold': 0.10,
    'min_epoch_s': 1.0,
    'min_active_reps': 2,
    'expected_reps': 4,
    'require_expected_reps': True,
}
REPORT_SETTINGS = {
    'save_report': False,
    'run_label': 'exp7_recruitment_containment',
    'report_format': 'html',
    'export_notebook': True,
    'fallback_to_html': True,
    'comments': 'Recruitment, containment and family breadth; enable saving after review.',
}


## 3. Independent loading and preflight

Load all five fish, validate the configured stimuli, and show the selected roots. The analysis step also validates the named groups and filtered-raster/continuous-trace row contract for every fish.


In [ ]:
raster_inputs = several_loading.load_and_preflight_fish_raster_inputs(USER_SETTINGS)
globals().update(raster_inputs)
print('Data root:', main_path)
print('Analysis root:', analysis_path)
print(f'Loaded {len(fish_ids)} fish.')


## 4. Build aligned analysis inputs once

Recruitment uses the canonical significant-raster activity decision. Continuous activity uses signed z-score AUC across the **same kept neurons**, including inactive neurons, divided by the actual trapezoidal integration span. This is response-window z-score activity, with no additional baseline subtraction or clipping at zero.

The existing timing helper starts each window 8 s after stimulus onset and requests stimulus motion duration plus `2 * tau_s`. It clips to the available aligned trace. The table below makes requested and observed durations explicit; conclusions concern these observed windows. Raw AUC, sample bounds, individual-neuron responses, and fish identities remain in `analysis['tables']`.


In [ ]:
analysis = several_diagnostics.build_recruitment_containment_analysis(
    all_fish_data, fish_ids, SIDE_GROUPS, timing, ACTIVE_SETTINGS, CONDITION_LABELS,
)
display(analysis['tables']['window_summary'])


## 5. Recruitment and response-window activity

Four separate figures show recruitment and response-window activity for left and right stimuli. Recruitment is the fraction of kept neurons active; activity is mean normalized AUC across that same population. Gray dots show fish values without identities or connecting lines; the black line and diamonds show their equally weighted mean. Both sides share the corresponding y scale. The activity panel combines recruitment and response magnitude; it does not establish stronger activity specifically within shared active neurons.


In [ ]:
# Each figure is independent; edit these dimensions for this section.
PANEL_WIDTH_MM, PANEL_HEIGHT_MM = 45, 27.5
FIGSIZE = (PANEL_WIDTH_MM * MM, PANEL_HEIGHT_MM * MM)
recruitment_panels = several_figures.plot_recruitment_activity_summary(
    analysis, figsize=FIGSIZE,
    separate=True, individual_style='anonymous_points',
)
for figure, axis in recruitment_panels.values():
    display(figure)
    plt.close(figure)


## 7. Containment in control and in all rocking variants

The control-containment figures show the fraction of each weaker-stimulus population also responding to its side's control. The all-rocking-containment figures show the fraction of each flicker population responding to **every rocking variant**, using their intersection, not their union. The denominator is the source stimulus's active population. Empty sources leave gaps. Non-contained counts and fractions are retained in `analysis['tables']['containment_summary']`.


In [ ]:
# Each figure is independent; edit these dimensions for this section.
PANEL_WIDTH_MM, PANEL_HEIGHT_MM = 45, 27.5
FIGSIZE = (PANEL_WIDTH_MM * MM, PANEL_HEIGHT_MM * MM)
containment_panels = several_figures.plot_recruitment_containment_summary(
    analysis, figsize=FIGSIZE,
    separate=True, individual_style='anonymous_points',
)
for figure, axis in containment_panels.values():
    display(figure)
    plt.close(figure)


## 8. Prepare shared response populations for family breadth

Keep the existing shared preparation for the all-kept and all-rocking populations
required by section 12. It reuses the signed duration-normalized AUC and existing
activity decisions without reloading or reclassifying neurons. The table reports
population sizes. Similarity plots and pairwise similarity report tables are
omitted from this notebook.


In [ ]:
similarity = several_diagnostics.build_response_similarity_comparison(analysis)
display(similarity['tables']['response_similarity_population_summary'])


## 12. Prepare family response breadth

Reuse the prepared activity decisions and signed normalized AUC, without reloading, reclassifying, or normalizing again. Within **rocking and flicker separately**, record each neuron's exact membership and number of active variants. Base tables retain all-kept denominators, including nonresponders. Additional family-responsive tables exclude zero-active categories and divide each count by the neurons active to at least one variant of that family, separately within each fish and side; sections 13–14 use these conditional fractions. Fish with no family-responsive neurons have undefined fractions and do not contribute to the mean. Summaries average per-fish fractions equally; complete counts, denominators, and valid-fish counts remain in `breadth_profiles['tables']`.


In [ ]:
breadth_profiles = several_diagnostics.build_response_breadth_profiles(analysis, similarity)


## 13. Response breadth within each stimulus family

Four separate figures show rocking/flicker for left/right stimuli. Among neurons active to at least one variant of the plotted family, each belongs to exactly one category: active to 1, 2, or all 3 variants. Fractions sum to one within each fish/side/family with responders; zero-active neurons are excluded. Gray unconnected dots show individual fish values; the black line and diamonds show their equally weighted mean. These are threshold-based response memberships, not independently validated selectivity classes.


In [ ]:
# Each figure is independent; edit these dimensions for this section.
PANEL_WIDTH_MM, PANEL_HEIGHT_MM = 45, 27.5
FIGSIZE = (PANEL_WIDTH_MM * MM, PANEL_HEIGHT_MM * MM)
breadth_panels = several_figures.plot_response_breadth_summary(
    breadth_profiles, mode='breadth', figsize=FIGSIZE,
    separate=True, individual_style='anonymous_points', denominator='family_responsive',
)
for figure, axis in breadth_panels.values():
    display(figure)
    plt.close(figure)


## 14. Exact stimulus-response combinations

The seven nonempty, mutually exclusive combinations separate family-responsive neurons responding to one particular variant, one particular pair, or all three. Every category is present even when its count is zero. Fractions use **neurons active to at least one variant of the plotted family**, separately within each fish and side, so a small exclusive category is not automatically evidence of specific tuning; threshold crossings and trial variability remain possible explanations.

For compact labels, each three-digit code follows the variant order printed on the x axis: `1` means active and `0` means inactive. For example, `101` means the first and third variants, and `000` is excluded because it represents no response to this family. Category order and values are unchanged. Gray dots show fish; the black line shows their equal-fish mean.


In [ ]:
# Each figure is independent; edit these dimensions for this section.
PANEL_WIDTH_MM, PANEL_HEIGHT_MM = 45, 27.5
FIGSIZE = (PANEL_WIDTH_MM * MM, PANEL_HEIGHT_MM * MM)
combination_panels = several_figures.plot_response_breadth_summary(
    breadth_profiles, mode='combinations', figsize=FIGSIZE,
    separate=True, individual_style='anonymous_points', denominator='family_responsive',
)
for figure, axis in combination_panels.values():
    display(figure)
    plt.close(figure)


## 21. Review before polishing individual figures

Compare recruitment and containment with family response breadth and exact
combinations. Check whether differences recur across fish and sides. Keep
threshold-based membership distinct from response strength: exclusive memberships
do not establish reliable stimulus specificity. Independent-trial preference
validation remains deferred.


## 22. Optional reviewed-run report

Saving is disabled by default. Save the notebook with current outputs before enabling the existing report exporter.


In [ ]:
notebook_path = repo_root / 'notebooks' / 'calcium' / 'exp_07_trans_stat' / '02_response_overlap_and_selectivity.ipynb'
report_result = several_reporting.save_analysis_report_run(
    settings=settings, report_settings=REPORT_SETTINGS, analysis_path=analysis_path,
    tables={
        **{name: analysis['tables'][name] for name in (
            'neuron_responses', 'response_windows', 'window_summary',
            'recruitment', 'containment_summary',
        )},
        **{name: table for name, table in breadth_profiles['tables'].items()
           if name != 'response_profile_rows'},
    },
    notebook_path=notebook_path,
    report_name='02_response_overlap_and_selectivity_report',
    extra_metadata={
        'side_groups': SIDE_GROUPS, 'condition_labels': CONDITION_LABELS,
        'active_settings': ACTIVE_SETTINGS, 'data_root': main_path,
        'analysis_root': analysis_path, 'aggregation': 'equal_fish_weight',
        'breadth_denominator': 'family_responsive_neurons_per_fish_side',
        'breadth_base_tables_denominator': 'all_kept_neurons',
    },
)


## Export reviewed publication figures

Run after the plotting cells; the style cell controls whether files are written.


In [ ]:
publication_figures = {
    **{f'05_{name}': figure for name, (figure, axis) in recruitment_panels.items()},
    **{f'07_{name}': figure for name, (figure, axis) in containment_panels.items()},
    **{f'13_{name}': figure for name, (figure, axis) in breadth_panels.items()},
    **{f'14_{name}': figure for name, (figure, axis) in combination_panels.items()},
}
if EXPORT_PUBLICATION_FIGURES:
    for name, figure in publication_figures.items():
        if figure is not None:
            plot_common.save_publication_figure(
                figure, PUBLICATION_DIR / name,
                formats=PUBLICATION_FORMATS, dpi=PUBLICATION_DPI,
            )
    print('Publication figures:', PUBLICATION_DIR)
